# Hafta 7: Ensemble Learning ve Model Değerlendirme

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Bagging kavramı ve Bootstrap örnekleme
2. Random Forest modeli ve Feature Importance
3. AdaBoost (Adaptive Boosting)
4. Gradient Boosting ve XGBoost
5. Tek ağaç vs Ensemble karşılaştırması
6. ROC Curve, Precision-Recall Curve, Learning Curve
7. Cross-Validation stratejileri
8. Hiperparametre Optimizasyonu (GridSearch & RandomizedSearch)
9. Stacking Ensemble
10. Sonuç ve Yorumlar

## 1. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, BaggingClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, learning_curve)
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)

# XGBoost opsiyonel
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost başarıyla yüklendi!")
except ImportError:
    HAS_XGB = False
    print("XGBoost yüklü değil – pip install xgboost ile kurabilirsiniz.")
    print("XGBoost olmadan devam edilecek.")

print("Kütüphaneler başarıyla yüklendi!")

## 2. Teorik Arka Plan: Bootstrap ve Bagging

### Bootstrap Örnekleme
Orijinal veri setinden **yerine koyarak** rastgele örnekler çekmek.  
Her bootstrap sample, orijinal veriyle aynı boyuttadır ama bazı örnekler tekrar eder, bazıları hiç seçilmez.

### Bagging (Bootstrap Aggregating)
1. B adet bootstrap sample oluştur
2. Her sample üzerinde bağımsız bir model eğit
3. Tahminleri birleştir (çoğunluk oylaması veya ortalama)

$$\hat{f}_{bag}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(x)$$

In [ ]:
# Bootstrap örnekleme gösterimi
np.random.seed(42)
original_data = np.arange(1, 11)  # [1, 2, ..., 10]

print("Orijinal veri:", original_data)
print("\nBootstrap örnekleri:")
print("-" * 50)

for i in range(5):
    sample = np.random.choice(original_data, size=len(original_data), replace=True)
    oob = set(original_data) - set(sample)  # Out-of-bag örnekler
    print(f"  Sample {i+1}: {sorted(sample)}")
    print(f"           OOB (seçilmeyenler): {sorted(oob)}")

# OOB oranı: Bir örneğin seçilmeme olasılığı ≈ (1 - 1/n)^n ≈ 1/e ≈ 0.368
print(f"\nTeori: Her bootstrap'ta bir örneğin seçilmeme olasılığı ≈ {(1 - 1/10)**10:.3f}")
print(f"Yaklaşık OOB oranı: 1/e ≈ {1/np.e:.3f} (%{100/np.e:.1f})")

## 3. Veri Seti Yükleme ve Keşif

**Diabetes (Diyabet)** veri seti kullanılacaktır.  
Hedef değişken: `Outcome` — 1 = Diyabetik, 0 = Sağlıklı

| Öznitelik | Açıklama |
|-----------|----------|
| Pregnancies | Hamilelik sayısı |
| Glucose | Glikoz seviyesi |
| BloodPressure | Kan basıncı |
| SkinThickness | Deri kalınlığı |
| Insulin | İnsülin seviyesi |
| BMI | Vücut kitle indeksi |
| DiabetesPedigreeFunction | Diyabet soy ağacı fonksiyonu |
| Age | Yaş |
| Outcome | Diyabet (1=Evet, 0=Hayır) |

In [ ]:
import os

DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         '..', '..', 'Veri_Setleri_Genel', 'diabetes.csv')

df = pd.read_csv(DATA_PATH)
print(f"Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
print("\nİlk 5 satır:")
df.head()

In [ ]:
print("Temel istatistikler:")
print(df.describe().round(2).to_string())
print(f"\nEksik değer sayıları:\n{df.isnull().sum()}")
print(f"\nHedef değişken dağılımı:\n{df['Outcome'].value_counts()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sınıf dağılımı
counts = df['Outcome'].value_counts()
axes[0].bar(['Sağlıklı (0)', 'Diyabetik (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Hedef Değişken Dağılımı')
axes[0].set_ylabel('Örnek Sayısı')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Glucose dağılımı
sns.histplot(data=df, x='Glucose', hue='Outcome', kde=True, ax=axes[1],
             palette=['steelblue', 'tomato'])
axes[1].set_title('Glikoz Seviyesine Göre Dağılım')

# Korelasyon
corr = df.corr()
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[2], mask=mask, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[2].set_title('Korelasyon Matrisi')

plt.tight_layout()
plt.show()

## 4. Veri Ön İşleme ve Eğitim / Test Ayrımı

In [ ]:
feature_cols = [c for c in df.columns if c != 'Outcome']
X = df[feature_cols].values
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Ölçeklendirme (Stacking içindeki modeller için)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Toplam örnek  : {len(X)}")
print(f"Eğitim seti   : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test seti     : {len(X_test)}   ({len(X_test)/len(X)*100:.1f}%)")
print(f"Öznitelik sayısı: {X.shape[1]}")
print(f"\nEğitim – sınıf dağılımı : {np.bincount(y_train)}")
print(f"Test   – sınıf dağılımı : {np.bincount(y_test)}")

---
## 5. Baseline: Tek Karar Ağacı

In [ ]:
# Baseline: Tek karar ağacı
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print("=" * 50)
print("Baseline: Tek Karar Ağacı (max_depth=5)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {dt_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_dt):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_dt,
                            target_names=['Sağlıklı', 'Diyabetik']))

---
## 6. Bagging Classifier

Karar ağaçlarını bootstrap samples üzerinde eğitip birleştiriyoruz.

In [ ]:
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=100,
    max_samples=1.0,     # Her bootstrap sample orijinal boyutta
    bootstrap=True,
    oob_score=True,      # Out-of-bag error
    random_state=42,
    n_jobs=-1
)
bag_model.fit(X_train, y_train)
y_pred_bag = bag_model.predict(X_test)

print("=" * 50)
print("Bagging Classifier (100 ağaç)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {bag_model.score(X_train, y_train):.4f}")
print(f"OOB Doğruluğu    : {bag_model.oob_score_:.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_bag):.4f}")

---
## 7. Random Forest

**Bagging + Öznitelik rastgeleliği:** Her split'te $\sqrt{n}$ öznitelik rastgele seçilir.

Bu ek rastgelelik, ağaçlar arasındaki korelasyonu azaltarak ensemble'ın varyansını düşürür.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=" * 50)
print("Random Forest (200 ağaç, max_depth=10)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {rf_model.score(X_train, y_train):.4f}")
print(f"OOB Doğruluğu    : {rf_model.oob_score_:.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_rf,
                            target_names=['Sağlıklı', 'Diyabetik']))

In [ ]:
# Random Forest: Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_cols)))
plt.bar(range(len(feature_cols)), importances[indices],
        color=colors, edgecolor='black')
plt.xticks(range(len(feature_cols)),
           [feature_cols[i] for i in indices], rotation=35, ha='right')
plt.title('Random Forest – Öznitelik Önemi', fontsize=13, fontweight='bold')
plt.ylabel('Önem Skoru')
plt.tight_layout()
plt.show()

print("Önem sıralaması:")
for i in indices:
    print(f"  {feature_cols[i]:<30} : {importances[i]:.4f}")

In [ ]:
# n_estimators (ağaç sayısı) etkisi
n_trees = [1, 5, 10, 25, 50, 100, 200, 300, 500]
oob_scores = []
test_scores = []

for n in n_trees:
    rf_temp = RandomForestClassifier(n_estimators=n, max_depth=10,
                                     oob_score=True, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    test_scores.append(rf_temp.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_trees, oob_scores, 'b-o', label='OOB Accuracy', markersize=5)
plt.plot(n_trees, test_scores, 'r-s', label='Test Accuracy', markersize=5)
plt.xlabel('Ağaç Sayısı (n_estimators)')
plt.ylabel('Accuracy')
plt.title('Random Forest: Ağaç Sayısının Etkisi', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_n = n_trees[np.argmax(test_scores)]
print(f"En iyi test accuracy → n_estimators = {best_n} ({max(test_scores):.4f})")

---
## 8. AdaBoost (Adaptive Boosting)

Her iterasyonda **yanlış sınıflandırılan örneklere daha fazla ağırlık** verilerek yeni bir zayıf öğrenici eğitilir.

$$F_M(x) = \sum_{m=1}^{M} \alpha_m \cdot h_m(x)$$

burada $\alpha_m$ model ağırlığı, $h_m$ zayıf öğrenicidir.

In [ ]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision stump
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)
ada_model.fit(X_train, y_train)
y_pred_ada = ada_model.predict(X_test)

print("=" * 50)
print("AdaBoost (200 stump, lr=0.1)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {ada_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_ada):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_ada,
                            target_names=['Sağlıklı', 'Diyabetik']))

In [ ]:
# AdaBoost: n_estimators ve learning_rate etkisi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: n_estimators etkisi
n_est_values = [10, 25, 50, 100, 200, 300, 500]
ada_train, ada_test = [], []
for n in n_est_values:
    ada_temp = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n, learning_rate=0.1, random_state=42)
    ada_temp.fit(X_train, y_train)
    ada_train.append(ada_temp.score(X_train, y_train))
    ada_test.append(ada_temp.score(X_test, y_test))

axes[0].plot(n_est_values, ada_train, 'b-o', label='Eğitim', markersize=5)
axes[0].plot(n_est_values, ada_test, 'r-s', label='Test', markersize=5)
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('AdaBoost: n_estimators Etkisi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sağ: learning_rate etkisi
lr_values = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
lr_train, lr_test = [], []
for lr in lr_values:
    ada_temp = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=200, learning_rate=lr, random_state=42)
    ada_temp.fit(X_train, y_train)
    lr_train.append(ada_temp.score(X_train, y_train))
    lr_test.append(ada_temp.score(X_test, y_test))

axes[1].plot(range(len(lr_values)), lr_train, 'b-o', label='Eğitim', markersize=5)
axes[1].plot(range(len(lr_values)), lr_test, 'r-s', label='Test', markersize=5)
axes[1].set_xticks(range(len(lr_values)))
axes[1].set_xticklabels([str(lr) for lr in lr_values])
axes[1].set_xlabel('learning_rate')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('AdaBoost: Learning Rate Etkisi')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Gradient Boosting

Her iterasyonda **residual (artık hata)** üzerine yeni bir model eğitilir:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

burada $\eta$ (learning_rate) küçültme faktörü, $h_m$ residual'a fit edilen ağaçtır.

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("=" * 50)
print("Gradient Boosting (200 ağaç, lr=0.1, depth=3)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {gb_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_gb):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_gb,
                            target_names=['Sağlıklı', 'Diyabetik']))

In [ ]:
# Gradient Boosting: Staged predictions – iterasyon başına performans
train_staged = list(gb_model.staged_score(X_train, y_train))
test_staged = list(gb_model.staged_score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_staged) + 1), train_staged, 'b-', label='Eğitim', alpha=0.7)
plt.plot(range(1, len(test_staged) + 1), test_staged, 'r-', label='Test', alpha=0.7)
plt.xlabel('Boosting İterasyonu')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting: İterasyon Başına Performans', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_iter = np.argmax(test_staged) + 1
print(f"En iyi test accuracy → iterasyon {best_iter} ({max(test_staged):.4f})")

---
## 10. XGBoost (Extreme Gradient Boosting)

Gradient Boosting'in gelişmiş versiyonu:  
- L1/L2 regularization  
- Paralel ve dağıtık hesaplama  
- Eksik değer yönetimi  
- Tree pruning (max_depth ile)

In [ ]:
if HAS_XGB:
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        reg_alpha=0.1,       # L1 regularization
        reg_lambda=1.0,      # L2 regularization
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    print("=" * 50)
    print("XGBoost (200 ağaç, lr=0.1, depth=5)")
    print("=" * 50)
    print(f"Eğitim Doğruluğu : {xgb_model.score(X_train, y_train):.4f}")
    print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_xgb):.4f}")
    print("\nSınıflandırma Raporu:")
    print(classification_report(y_test, y_pred_xgb,
                                target_names=['Sağlıklı', 'Diyabetik']))

    # XGBoost Feature Importance
    xgb_imp = xgb_model.feature_importances_
    xgb_idx = np.argsort(xgb_imp)[::-1]
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(feature_cols)), xgb_imp[xgb_idx],
            color=plt.cm.magma(np.linspace(0.2, 0.8, len(feature_cols))),
            edgecolor='black')
    plt.xticks(range(len(feature_cols)),
               [feature_cols[i] for i in xgb_idx], rotation=35, ha='right')
    plt.title('XGBoost – Öznitelik Önemi', fontsize=13, fontweight='bold')
    plt.ylabel('Önem Skoru')
    plt.tight_layout()
    plt.show()
else:
    print("XGBoost yüklü olmadığı için bu bölüm atlanıyor.")
    y_pred_xgb = None

---
## 11. Tüm Modellerin Karşılaştırması

In [ ]:
# Confusion Matrix – yan yana
all_predictions = {
    'Tek Ağaç': y_pred_dt,
    'Bagging': y_pred_bag,
    'Random Forest': y_pred_rf,
    'AdaBoost': y_pred_ada,
    'Gradient Boosting': y_pred_gb,
}
if HAS_XGB:
    all_predictions['XGBoost'] = y_pred_xgb

n_models = len(all_predictions)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))

for ax, (name, y_pred) in zip(axes, all_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Sağlıklı', 'Diyabetik'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=10)

plt.suptitle('Karmaşıklık Matrisleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Metrik karşılaştırma tablosu
results = []
for name, y_pred in all_predictions.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print("Model Karşılaştırma Tablosu:")
print("=" * 70)
print(results_df.round(4).to_string())

# Görselleştirme
fig, ax = plt.subplots(figsize=(14, 6))
x_pos = np.arange(len(results_df.index))
width = 0.2
colors = ['steelblue', 'seagreen', 'tomato', 'darkorange']

for i, metric in enumerate(results_df.columns):
    bars = ax.bar(x_pos + i * width, results_df[metric], width,
                  label=metric, color=colors[i], edgecolor='black', alpha=0.85)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15, ha='right')
ax.set_ylabel('Skor')
ax.set_ylim(0, 1.15)
ax.set_title('Ensemble Modelleri – Performans Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 12. ROC Eğrileri Karşılaştırması

In [ ]:
# ROC eğrileri
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

roc_models = {
    'Tek Ağaç': dt_model,
    'Random Forest': rf_model,
    'AdaBoost': ada_model,
    'Gradient Boosting': gb_model,
}
if HAS_XGB:
    roc_models['XGBoost'] = xgb_model

colors_roc = ['gray', 'steelblue', 'darkorange', 'seagreen', 'tomato', 'purple']

# Sol: ROC Curve
for (name, model), color in zip(roc_models.items(), colors_roc):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                label=f'{name} (AUC={roc_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Eğrileri', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Sağ: Precision-Recall Curve
for (name, model), color in zip(roc_models.items(), colors_roc):
    y_proba = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    axes[1].plot(rec, prec, color=color, linewidth=2,
                label=f'{name} (AP={ap:.3f})')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Eğrileri', fontweight='bold')
axes[1].legend(loc='lower left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 13. Learning Curve – Veri Miktarının Etkisi

Model, daha fazla eğitim verisi ile nasıl öğreniyor?  
- **Eğitim ve test eğrisi yakınsa:** İyi genelleme  
- **Eğitim yüksek, test düşükse:** Overfitting  
- **İkisi de düşükse:** Underfitting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lc_models = {
    'Tek Ağaç (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                            random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                    learning_rate=0.1, random_state=42),
}

for ax, (name, model) in zip(axes, lc_models.items()):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    test_mean = test_scores.mean(axis=1)
    test_std = test_scores.std(axis=1)

    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.1, color='blue')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                    alpha=0.1, color='red')
    ax.plot(train_sizes, train_mean, 'b-o', markersize=4, label='Eğitim')
    ax.plot(train_sizes, test_mean, 'r-s', markersize=4, label='CV Test')
    ax.set_xlabel('Eğitim Örnek Sayısı')
    ax.set_ylabel('Accuracy')
    ax.set_title(name, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. Cross-Validation ile Adil Karşılaştırma

In [ ]:
# 5-Katlı Stratified Cross-Validation
cv_models = {
    'Tek Ağaç': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10,
                                            random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.1,
                                   random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                    learning_rate=0.1, random_state=42),
}
if HAS_XGB:
    cv_models['XGBoost'] = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1,
                                              max_depth=5, random_state=42,
                                              use_label_encoder=False,
                                              eval_metric='logloss')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-Katlı Stratified CV Sonuçları:")
print("=" * 65)
print(f"{'Model':<22} {'Ortalama':>10} {'Std Dev':>10} {'Min':>8} {'Max':>8}")
print("-" * 65)

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<22} {scores.mean():>9.4f} {scores.std():>10.4f}"
          f" {scores.min():>7.4f} {scores.max():>7.4f}")

In [ ]:
# CV sonuçları boxplot
fig, ax = plt.subplots(figsize=(12, 5))
bp_data = [cv_results[name] for name in cv_results]
bp = ax.boxplot(bp_data, labels=list(cv_results.keys()), patch_artist=True)

bp_colors = ['gray', 'lightblue', 'steelblue', 'darkorange', 'seagreen', 'tomato']
for patch, color in zip(bp['boxes'], bp_colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy')
ax.set_title('5-Katlı Stratified CV – Model Karşılaştırması', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## 15. Hiperparametre Optimizasyonu

### GridSearchCV vs RandomizedSearchCV
| Yöntem | Avantaj | Dezavantaj |
|--------|---------|------------|
| GridSearch | Tüm kombinasyonları dener | Yavaş (büyük grid) |
| RandomizedSearch | Hızlı, rastgele dener | En iyiyi garanti etmez |

In [ ]:
# Random Forest – GridSearchCV
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print("Random Forest – GridSearchCV")
print("=" * 50)
print("En iyi parametreler:")
for k, v in rf_grid.best_params_.items():
    print(f"  {k:<22} : {v}")
print(f"\nEn iyi CV Accuracy : {rf_grid.best_score_:.4f}")
print(f"Test Accuracy      : {rf_grid.score(X_test, y_test):.4f}")

In [ ]:
# Gradient Boosting – RandomizedSearchCV
from scipy.stats import randint, uniform

gb_param_dist = {
    'n_estimators': randint(50, 500),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 10),
    'min_samples_split': randint(2, 20),
    'subsample': uniform(0.6, 0.4),
}

gb_random = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_dist, n_iter=30, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1
)
gb_random.fit(X_train, y_train)

print("Gradient Boosting – RandomizedSearchCV (30 iterasyon)")
print("=" * 55)
print("En iyi parametreler:")
for k, v in gb_random.best_params_.items():
    if isinstance(v, float):
        print(f"  {k:<22} : {v:.4f}")
    else:
        print(f"  {k:<22} : {v}")
print(f"\nEn iyi CV Accuracy : {gb_random.best_score_:.4f}")
print(f"Test Accuracy      : {gb_random.score(X_test, y_test):.4f}")

---
## 16. Stacking Ensemble

Farklı modellerin çıktılarını bir **meta-model** (Logistic Regression) ile birleştiriyoruz.

In [ ]:
# Stacking: farklı base modellerden oluşan ensemble
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                      learning_rate=0.1, random_state=42)),
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)
stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("=" * 50)
print("Stacking Ensemble (RF + AdaBoost + GB → LR)")
print("=" * 50)
print(f"Eğitim Doğruluğu : {stack_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_stack):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_stack,
                            target_names=['Sağlıklı', 'Diyabetik']))

---
## 17. Final: Tüm Modellerin Özeti

In [ ]:
# Final karşılaştırma
final_models = {
    'Tek Ağaç (baseline)': accuracy_score(y_test, y_pred_dt),
    'Bagging': accuracy_score(y_test, y_pred_bag),
    'Random Forest': accuracy_score(y_test, y_pred_rf),
    'RF (GridSearch)': rf_grid.score(X_test, y_test),
    'AdaBoost': accuracy_score(y_test, y_pred_ada),
    'Gradient Boosting': accuracy_score(y_test, y_pred_gb),
    'GB (RandomSearch)': gb_random.score(X_test, y_test),
    'Stacking': accuracy_score(y_test, y_pred_stack),
}
if HAS_XGB:
    final_models['XGBoost'] = accuracy_score(y_test, y_pred_xgb)

print("\n" + "=" * 55)
print("FINAL SONUÇLAR – Test Seti Accuracy")
print("=" * 55)

# Sıralı
sorted_models = sorted(final_models.items(), key=lambda x: x[1], reverse=True)
for rank, (name, score) in enumerate(sorted_models, 1):
    bar = '█' * int(score * 40)
    marker = ' ← EN İYİ' if rank == 1 else ''
    print(f"  {rank}. {name:<24} : {score:.4f}  {bar}{marker}")

# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 6))
names = [s[0] for s in sorted_models]
scores = [s[1] for s in sorted_models]
colors_final = plt.cm.RdYlGn(np.linspace(0.8, 0.3, len(names)))

bars = ax.barh(range(len(names)), scores, color=colors_final, edgecolor='black')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Test Accuracy')
ax.set_title('Ensemble Modelleri – Final Sıralama', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.invert_yaxis()

for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(score + 0.005, i, f'{score:.4f}', va='center', fontsize=9)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 18. Sonuç ve Yorumlar

### Ders Çıkarımları

| Yöntem | Strateji | Ne Azaltır? | Önemli Parametre |
|--------|---------|-------------|------------------|
| **Bagging** | Paralel modeller | Varyans | n_estimators |
| **Random Forest** | Bagging + öznitelik rastgeleliği | Varyans | n_estimators, max_features |
| **AdaBoost** | Sıralı, hatalara odaklan | Bias | n_estimators, learning_rate |
| **Gradient Boosting** | Sıralı, residual'a fit | Bias | n_estimators, learning_rate, max_depth |
| **XGBoost** | GB + regularization | Bias + Varyans | Tüm GB + reg_alpha, reg_lambda |
| **Stacking** | Farklı modelleri birleştir | Genel hata | Base learners, meta-learner |

### Pratik İpuçları
1. **Her zaman baseline ile başla** (tek karar ağacı)
2. **Random Forest** güvenli bir başlangıç noktasıdır
3. **Gradient Boosting / XGBoost** genellikle en yüksek performansı verir
4. **Cross-validation** kullanarak model seçimi yap
5. **Hiperparametre optimizasyonu** son adımda yapılmalı
6. **Overfitting kontrolü:** Eğitim-test farkını izle